# Stage 3.5 — Sanity-check the trained IQL policy

Loads the TorchScript policy (`data/policy.pt`) + the combined embedding
(`data/game_embeddings_matrix.npy` / `_index.pkl`) and prints the top-10
recommendations the policy produces from a cold-start state and four
distinct user profiles (RPG / FPS / strategy / indie).

**This is a human gate before Stage 4.** Eyeball whether the policy adapts
its recommendations to the user's history and whether cold-start picks are
plausible. The last code cell is left empty for ad-hoc queries.

The policy emits a continuous *action vector* (dim 1584); recommendations
are the catalog games whose embedding is closest (cosine) to that vector,
with the user's own played games filtered out — the same flow Stage 5
(`inference.py`) will use.


In [1]:
import pickle
import numpy as np
import pandas as pd
import torch
from thefuzz import fuzz

E = np.load("../../data/game_embeddings_matrix.npy")
index_df = pd.read_pickle("../../data/game_embeddings_index.pkl")
names = index_df["name"].tolist()
name_to_row = dict(zip(index_df["name"].values, index_df["row_idx"].values))

# Load the TorchScript policy on CPU. The policy was traced on cuda:0 during
# training, so map_location="cpu" is required to run it without a GPU (this is
# also how the HuggingFace Space will load it).
policy = torch.jit.load("../../data/policy.pt", map_location="cpu")
policy.eval()

# Pre-normalize E once for cosine scoring.
E_norm = E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-12)

print(f"Loaded policy + E {E.shape}, {len(names)} games")


Loaded policy + E (26120, 1584), 26120 games


In [2]:
def resolve_name(query, min_ratio=80):
    """Fuzzy-resolve a (possibly user-typed) title to a canonical game name."""
    if query in name_to_row:
        return query
    best, best_score = None, -1
    q = query.lower()
    for n in names:
        s = fuzz.ratio(q, n.lower())
        if s > best_score:
            best, best_score = n, s
    return best if best_score >= min_ratio else None


def build_state(played_games):
    """Mean of E-vectors over recognized games; zero vector for cold start.

    Returns (state_vector, resolved_names). Mirrors the cold_start_state that
    Stage 6 will use, including fuzzy name resolution.
    """
    rows, resolved = [], []
    for g in played_games:
        canon = resolve_name(g)
        if canon is not None:
            rows.append(name_to_row[canon])
            resolved.append(canon)
    if not rows:
        return np.zeros(E.shape[1], dtype=np.float32), resolved
    return E[rows].mean(axis=0).astype(np.float32), resolved


In [3]:
def top_k_recommendations(state, top_n=10, played=None, verbose=True):
    """Run the policy on `state`; return top-N games by cosine to the predicted action."""
    played = set(played or [])
    t = torch.from_numpy(np.asarray(state, dtype=np.float32))
    if t.ndim == 1:
        t = t[None, :]
    with torch.no_grad():
        action = policy(t).cpu().numpy().ravel()
    a_norm = action / max(float(np.linalg.norm(action)), 1e-12)
    sims = E_norm @ a_norm
    results = []
    for idx in np.argsort(-sims):
        nm = names[idx]
        if nm in played:
            continue
        results.append((nm, float(sims[idx])))
        if len(results) == top_n:
            break
    if verbose:
        for i, (nm, sc) in enumerate(results, 1):
            print(f"{i:2d}. {sc:.4f}  {nm}")
    return results


def recommend_for(played_games, top_n=10):
    """Build a state from the played games and print the policy's top-N recs."""
    state, resolved = build_state(played_games)
    if resolved:
        print(f"History (resolved {len(resolved)}/{len(played_games)}): {resolved}")
    else:
        print("Cold start (empty history)")
    print("-" * 70)
    return top_k_recommendations(state, top_n=top_n, played=resolved)


## Cold start


In [4]:
recommend_for([])


Cold start (empty history)
----------------------------------------------------------------------
 1. 0.8256  Revenge of Arcade
 2. 0.8173  Bioshock Infinite: The Complete Edition
 3. 0.8169  BattleZone (2006)
 4. 0.8167  Rayman 2: Revolution
 5. 0.8149  WarGames
 6. 0.8147  Mission: Impossible 3
 7. 0.8129  Fury 3
 8. 0.8114  Space Invaders Extreme Z
 9. 0.8089  Wipeout 3
10. 0.8089  Dragon Fire 2


[('Revenge of Arcade', 0.8256170749664307),
 ('Bioshock Infinite: The Complete Edition', 0.8173149824142456),
 ('BattleZone (2006)', 0.8168665170669556),
 ('Rayman 2: Revolution', 0.816664457321167),
 ('WarGames', 0.8148667812347412),
 ('Mission: Impossible 3', 0.8146660327911377),
 ('Fury 3', 0.8129087090492249),
 ('Space Invaders Extreme Z', 0.8114075064659119),
 ('Wipeout 3', 0.8089269399642944),
 ('Dragon Fire 2', 0.8088884353637695)]

## RPG fan


In [5]:
recommend_for(['The Witcher 3: Wild Hunt', 'Dark Souls III', 'The Elder Scrolls V: Skyrim'])


History (resolved 3/3): ['The Witcher 3: Wild Hunt', 'Dark Souls III', 'The Elder Scrolls V: Skyrim']
----------------------------------------------------------------------
 1. 0.7711  Half-Life 2
 2. 0.7476  Battlefield Hardline: Getaway
 3. 0.7472  Marvel's Spider-Man - Game of the Year Edition
 4. 0.7469  Revenge of Arcade
 5. 0.7460  Battlefield Hardline: Betrayal
 6. 0.7434  Gothic II
 7. 0.7434  Mission: Impossible 3
 8. 0.7433  Underworld
 9. 0.7428  Cyberpunk 2077: Ultimate Edition
10. 0.7414  The Witcher: Enhanced Edition


[('Half-Life 2', 0.7711381912231445),
 ('Battlefield Hardline: Getaway', 0.747580885887146),
 ("Marvel's Spider-Man - Game of the Year Edition", 0.7471746802330017),
 ('Revenge of Arcade', 0.7469324469566345),
 ('Battlefield Hardline: Betrayal', 0.7460058927536011),
 ('Gothic II', 0.7433810234069824),
 ('Mission: Impossible 3', 0.7433810234069824),
 ('Underworld', 0.7433128356933594),
 ('Cyberpunk 2077: Ultimate Edition', 0.7427865862846375),
 ('The Witcher: Enhanced Edition', 0.741368293762207)]

## FPS fan


In [6]:
recommend_for(['DOOM Eternal', 'Halo Infinite', 'Titanfall 2'])


History (resolved 3/3): ['DOOM Eternal', 'Halo Infinite', 'Titanfall 2']
----------------------------------------------------------------------
 1. 0.8198  DOOM
 2. 0.8057  Revenge of Arcade
 3. 0.8041  Halo: Reach Remastered
 4. 0.8009  Prince of Persia: The Lost Crown - Mask of Darkness
 5. 0.8009  WarGames
 6. 0.7980  BattleZone (2006)
 7. 0.7976  Bioshock Infinite: The Complete Edition
 8. 0.7973  Little Nightmares: Complete Edition
 9. 0.7962  Mission: Impossible 3
10. 0.7950  Space Invaders Extreme Z


[('DOOM', 0.8198342323303223),
 ('Revenge of Arcade', 0.8057056069374084),
 ('Halo: Reach Remastered', 0.8041296005249023),
 ('Prince of Persia: The Lost Crown - Mask of Darkness', 0.8009068965911865),
 ('WarGames', 0.800864577293396),
 ('BattleZone (2006)', 0.7980125546455383),
 ('Bioshock Infinite: The Complete Edition', 0.797642707824707),
 ('Little Nightmares: Complete Edition', 0.7973074316978455),
 ('Mission: Impossible 3', 0.796218991279602),
 ('Space Invaders Extreme Z', 0.7950338125228882)]

## Strategy fan


In [7]:
recommend_for(["Sid Meier's Civilization VI", 'Total War: WARHAMMER'])


History (resolved 2/2): ["Sid Meier's Civilization VI", 'Total War: WARHAMMER']
----------------------------------------------------------------------
 1. 0.7701  Mount & Blade: Warband - Viking Conquest
 2. 0.7686  Age of Empires II: HD Edition
 3. 0.7667  The Witcher: Enhanced Edition
 4. 0.7611  Gothic II
 5. 0.7592  Sid Meier's Civilization IV: Beyond the Sword
 6. 0.7552  Resident Evil 5: Untold Stories
 7. 0.7548  Pathfinder: Wrath of the Righteous - Through the Ashes
 8. 0.7541  Heroes of Might and Magic III
 9. 0.7532  The Elder Scrolls IV: Knights of the Nine
10. 0.7509  Heroes Rise


[('Mount & Blade: Warband - Viking Conquest', 0.7700517177581787),
 ('Age of Empires II: HD Edition', 0.7685654163360596),
 ('The Witcher: Enhanced Edition', 0.7666684985160828),
 ('Gothic II', 0.7611403465270996),
 ("Sid Meier's Civilization IV: Beyond the Sword", 0.7591546773910522),
 ('Resident Evil 5: Untold Stories', 0.7551792860031128),
 ('Pathfinder: Wrath of the Righteous - Through the Ashes',
  0.7548485994338989),
 ('Heroes of Might and Magic III', 0.7541452646255493),
 ('The Elder Scrolls IV: Knights of the Nine', 0.7532005310058594),
 ('Heroes Rise', 0.7508672475814819)]

## Indie fan


In [8]:
recommend_for(['Hollow Knight', 'Celeste', 'Stardew Valley'])


History (resolved 3/3): ['Hollow Knight', 'Celeste', 'Stardew Valley']
----------------------------------------------------------------------
 1. 0.8112  DOOM
 2. 0.8050  Bulletstorm: Duke of Switch Edition
 3. 0.8045  Capcom Arcade Stadium: Ghosts 'n Goblins
 4. 0.8043  Revenge of Arcade
 5. 0.8021  Outlast: Bundle of Terror
 6. 0.8013  Mission: Impossible 3
 7. 0.7997  Bioshock Infinite: The Complete Edition
 8. 0.7982  Omori
 9. 0.7975  Fury 3
10. 0.7974  WarGames


[('DOOM', 0.8111935257911682),
 ('Bulletstorm: Duke of Switch Edition', 0.8050485253334045),
 ("Capcom Arcade Stadium: Ghosts 'n Goblins", 0.8045372366905212),
 ('Revenge of Arcade', 0.8043134212493896),
 ('Outlast: Bundle of Terror', 0.8021138906478882),
 ('Mission: Impossible 3', 0.8013361692428589),
 ('Bioshock Infinite: The Complete Edition', 0.7997032403945923),
 ('Omori', 0.7981890439987183),
 ('Fury 3', 0.7974605560302734),
 ('WarGames', 0.7974026203155518)]

## Profile divergence diagnostic

How much does the policy actually adapt to the user's history? This computes
the pairwise Jaccard overlap of the top-10 recommendation sets across the
profiles above. Low overlap = the policy differentiates; high overlap (near 1)
= the policy is collapsing to roughly the same recommendations regardless of
input, which would be a red flag worth more training / tuning before Stage 4.


In [9]:
_profiles = {
    'cold': build_state([])[0],
    'rpg': build_state(['The Witcher 3: Wild Hunt', 'Dark Souls III', 'The Elder Scrolls V: Skyrim'])[0],
    'fps': build_state(['DOOM Eternal', 'Halo Infinite', 'Titanfall 2'])[0],
    'strategy': build_state(["Sid Meier's Civilization VI", 'Total War: WARHAMMER'])[0],
    'indie': build_state(['Hollow Knight', 'Celeste', 'Stardew Valley'])[0],
}
_top10 = {k: {nm for nm, _ in top_k_recommendations(v, top_n=10, verbose=False)}
          for k, v in _profiles.items()}

_keys = list(_top10)
print(f"{'':10s}" + "".join(f"{k:>10s}" for k in _keys))
for a in _keys:
    row = f"{a:10s}"
    for b in _keys:
        j = len(_top10[a] & _top10[b]) / len(_top10[a] | _top10[b])
        row += f"{j:>10.2f}"
    print(row)
print("\n(1.00 on the diagonal; off-diagonal = Jaccard overlap of top-10 sets)")


                cold       rpg       fps  strategy     indie
cold            1.00      0.11      0.43      0.00      0.33
rpg             0.11      1.00      0.11      0.05      0.11
fps             0.43      0.11      1.00      0.00      0.33
strategy        0.00      0.05      0.00      1.00      0.00
indie           0.33      0.11      0.33      0.00      1.00

(1.00 on the diagonal; off-diagonal = Jaccard overlap of top-10 sets)


## Stage 4 — policy reranking over a filtered candidate set

The cells above run the policy over the **whole catalog**, which surfaced a
recurring "default cluster" (Revenge of Arcade, WarGames, BattleZone…) in the
weaker profiles. Stage 4's candidate generator is meant to mask that: it first
narrows the catalog by the user's filters (year / platform / language) and
optionally reranks by similarity to the played games, then the policy reranks
*within that shortlist*.

`recommend_with_candidates` below previews the full Stage 5 flow. For one
profile (RPG) we decompose the contribution:

- **A — full catalog**: policy over all 26k games (Stage 3.5 behaviour).
- **B — filter only**: `candidates(filters)` (no profile rerank) → policy rerank. Isolates what the policy does over a filtered-but-unsorted set.
- **C — full pipeline**: `candidates(filters, played_games)` (filter + cosine profile rerank) → policy rerank. What Stage 5 ships.


In [10]:
import sys
if ".." not in sys.path:
    sys.path.insert(0, "..")  # make the `app` package importable from source/scripts
from app.candidate_generator import candidates


def recommend_with_candidates(played_games, filters, top_n=10, use_profile_rerank=True, k=500, verbose=True):
    """Preview of Stage 5: candidate filter (+optional profile rerank) -> policy rerank -> drop played."""
    state, resolved = build_state(played_games)
    cg_played = resolved if use_profile_rerank else None
    cand_idx = candidates(filters, played_games=cg_played, k=k)

    t = torch.from_numpy(state[None, :].astype(np.float32))
    with torch.no_grad():
        action = policy(t).cpu().numpy().ravel()
    a = action / max(float(np.linalg.norm(action)), 1e-12)

    sims = E_norm[cand_idx] @ a
    played_set = set(resolved)
    ranked = ((cand_idx[j], float(sims[j])) for j in np.argsort(-sims))
    out = [(names[i], sc) for i, sc in ranked if names[i] not in played_set][:top_n]

    if verbose:
        print(f"History: {resolved or 'cold start'}")
        print(f"Filters: {filters}  ->  {len(cand_idx)} candidates")
        print("-" * 70)
        for i, (nm, sc) in enumerate(out, 1):
            print(f"{i:2d}. {sc:.4f}  {nm}")
    return out


### RPG fan — A (full catalog) vs B (filter only) vs C (full pipeline)

Filter: released 2010+, on PC.


In [11]:
_rpg = ["The Witcher 3: Wild Hunt", "Dark Souls III", "The Elder Scrolls V: Skyrim"]
_filt = {"year_min": 2010, "platforms": ["PC"]}

print("=== A. policy over FULL catalog ===")
recommend_for(_rpg)
print("\n=== B. filter only -> policy rerank ===")
recommend_with_candidates(_rpg, _filt, use_profile_rerank=False)
print("\n=== C. filter + profile rerank -> policy rerank (Stage 5) ===")
recommend_with_candidates(_rpg, _filt, use_profile_rerank=True, k=30)


=== A. policy over FULL catalog ===
History (resolved 3/3): ['The Witcher 3: Wild Hunt', 'Dark Souls III', 'The Elder Scrolls V: Skyrim']
----------------------------------------------------------------------
 1. 0.7711  Half-Life 2
 2. 0.7476  Battlefield Hardline: Getaway
 3. 0.7472  Marvel's Spider-Man - Game of the Year Edition
 4. 0.7469  Revenge of Arcade
 5. 0.7460  Battlefield Hardline: Betrayal
 6. 0.7434  Gothic II
 7. 0.7434  Mission: Impossible 3
 8. 0.7433  Underworld
 9. 0.7428  Cyberpunk 2077: Ultimate Edition
10. 0.7414  The Witcher: Enhanced Edition

=== B. filter only -> policy rerank ===
History: ['The Witcher 3: Wild Hunt', 'Dark Souls III', 'The Elder Scrolls V: Skyrim']
Filters: {'year_min': 2010, 'platforms': ['PC']}  ->  500 candidates
----------------------------------------------------------------------
 1. 0.7264  Age of Empires II: HD Edition
 2. 0.6972  ATOM RPG: Trudograd
 3. 0.6971  Age of Empires IV: Anniversary Edition
 4. 0.6913  A New Beginning
 5. 0.

[('The Witcher: Enhanced Edition', 0.741368293762207),
 ('Red Dead Redemption 2', 0.7100119590759277),
 ('Pathfinder: Wrath of the Righteous - Through the Ashes',
  0.7035192251205444),
 ('Fallout 4', 0.6957823038101196),
 ('Dark Souls II: Scholar of the First Sin', 0.6740177869796753),
 ('Dark Souls II', 0.6649524569511414),
 ('Torchlight II', 0.6543188691139221),
 ('Fallout: New Vegas', 0.6503992080688477),
 ('Pillars of Eternity', 0.6503637433052063),
 ('Fable Anniversary', 0.647350013256073)]

### Indie fan — full pipeline

The noisiest profile in Stage 3.5. Filter: 2015+ (any platform).


In [12]:
recommend_with_candidates(["Hollow Knight", "Celeste", "Stardew Valley"], {"year_min": 2015}, k=30)

History: ['Hollow Knight', 'Celeste', 'Stardew Valley']
Filters: {'year_min': 2015}  ->  30 candidates
----------------------------------------------------------------------
 1. 0.8021  Outlast: Bundle of Terror
 2. 0.7982  Omori
 3. 0.7920  Rayman Legends: Definitive Edition
 4. 0.7883  Little Nightmares: Complete Edition
 5. 0.7879  Slay The Princess - The Pristine Cut
 6. 0.7844  Amazing Thief (2014)
 7. 0.7745  Danganronpa V3: Killing Harmony - Anniversary Edition
 8. 0.7731  Yeeps: Hide and Seek
 9. 0.7659  Star Wars Outlaws: Gold Edition
10. 0.7655  Fortnite: LEGO Fortnite


[('Outlast: Bundle of Terror', 0.8021138906478882),
 ('Omori', 0.7981890439987183),
 ('Rayman Legends: Definitive Edition', 0.7919500470161438),
 ('Little Nightmares: Complete Edition', 0.7882577180862427),
 ('Slay The Princess - The Pristine Cut', 0.7878682017326355),
 ('Amazing Thief (2014)', 0.78438401222229),
 ('Danganronpa V3: Killing Harmony - Anniversary Edition', 0.7744624614715576),
 ('Yeeps: Hide and Seek', 0.7731175422668457),
 ('Star Wars Outlaws: Gold Edition', 0.7659415602684021),
 ('Fortnite: LEGO Fortnite', 0.7654716968536377)]

### Filters actually bind — FPS fan, Nintendo Switch only, 2019+

Every recommendation should be a Switch game released in 2019 or later (or have unknown year/platform — the filter is lenient on missing metadata). The check below prints each pick's actual year + platforms.


In [13]:
_recs = recommend_with_candidates(["DOOM Eternal", "Halo Infinite", "Titanfall 2"], {"year_min": 2019, "platforms": ["Nintendo Switch"]}, k=30)
print("\nFilter-compliance check:")
_rows = {nm: r for nm, r in zip(index_df["name"], index_df["row_idx"])}
for nm, _ in _recs:
    row = index_df.iloc[_rows[nm]]
    yr = row["release_year"]
    plats = [p for p in row["platforms"] if "switch" in str(p).lower()] or row["platforms"][:3]
    print(f"  {nm}: year={yr}, switch_platforms={plats}")

History: ['DOOM Eternal', 'Halo Infinite', 'Titanfall 2']
Filters: {'year_min': 2019, 'platforms': ['Nintendo Switch']}  ->  30 candidates
----------------------------------------------------------------------
 1. 0.8198  DOOM
 2. 0.8009  Prince of Persia: The Lost Crown - Mask of Darkness
 3. 0.7871  DOOM Eternal: The Ancient Gods Part Two
 4. 0.7799  Dying Light: Platinum Edition
 5. 0.7780  Quake Remastered
 6. 0.7669  Ion Fury: Aftershock
 7. 0.7370  Samurai Warriors 5
 8. 0.6896  BioShock 2 Remastered
 9. 0.6530  Evil Dead: The Game
10. 0.6511  Serious Sam HD:  The First Encounter

Filter-compliance check:
  DOOM: year=nan, switch_platforms=['Nintendo Switch']
  Prince of Persia: The Lost Crown - Mask of Darkness: year=nan, switch_platforms=['Nintendo Switch']
  DOOM Eternal: The Ancient Gods Part Two: year=nan, switch_platforms=['Nintendo Switch']
  Dying Light: Platinum Edition: year=nan, switch_platforms=['Nintendo Switch']
  Quake Remastered: year=nan, switch_platforms=['Ninte

## Stage 5 — `inference.recommend` (full pipeline + metadata + compliance)

`recommend(state, filters, played_games, top_n)` is the function the app calls:
candidate filter → policy rerank → drop played → enrich with year/cover/description.

**Default mode is profile rerank** (`profile_prefilter=True`, `candidate_k=30`): the
candidate generator keeps the 30 games closest to the play history and the policy
reranks those — anchoring results to history and masking the policy's undertraining.
Cold start (no history) falls back to filter-only so the policy reranks the whole
filtered set. Pass `profile_prefilter=False` to always let the policy rank the full
filtered set (the "trust the policy" mode — see the Battlefield Hardline cell below).

`compliance_check` prints each recommendation's actual year / platforms / language
against the filter so you can confirm the filter bound — and see which picks pass
only via the *lenient-on-missing* rule.



In [14]:
from app.inference import recommend


def show_recs(recs):
    for i, r in enumerate(recs, 1):
        yr = r["release_year"] if r["release_year"] is not None else "?"
        cov = "cover" if r["cover_url"] else "no-cover"
        print(f"{i:2d}. {r['score']:.4f}  {r['name']}  [{yr}, {cov}]")


def compliance_check(recs, filters):
    """Print each rec's year/platform/language vs the filter; flag lenient passes & violations."""
    ymin, ymax = filters.get("year_min"), filters.get("year_max")
    req_plat = [p.lower() for p in (filters.get("platforms") or [])]
    req_lang = (filters.get("language") or "").lower()
    print(f"Filter: {filters}")
    for r in recs:
        row = index_df.iloc[name_to_row[r["name"]]]
        yr, plats, langs = row["release_year"], list(row["platforms"]), list(row["language_supports"])
        notes = []
        if ymin is not None or ymax is not None:
            if np.isnan(yr):
                notes.append("year=NaN(lenient)")
            else:
                ok = (ymin is None or yr >= ymin) and (ymax is None or yr <= ymax)
                notes.append(f"year={int(yr)}{'' if ok else ' !VIOLATION'}")
        if req_plat:
            if not plats:
                notes.append("no-platforms(lenient)")
            else:
                hit = [p for p in plats if any((rp in str(p).lower()) or (str(p).lower() in rp) for rp in req_plat)]
                notes.append(f"plat={hit}" if hit else f"!PLATFORM VIOLATION {plats[:3]}")
        if req_lang:
            if not langs:
                notes.append("no-language(lenient)")
            else:
                ok = any(req_lang in str(l).lower() for l in langs)
                notes.append("lang-ok" if ok else "!LANG VIOLATION")
        print(f"  {r['name']}: {'; '.join(notes) if notes else '(no filter)'}")


### Scenario 1 — RPG fan · 2010+ · PC · default (profile rerank, k=30)



In [ ]:
_rpg = ["The Witcher 3: Wild Hunt", "Dark Souls III", "The Elder Scrolls V: Skyrim"]
_f = {"year_min": 2010, "platforms": ["PC"]}
_s, _resolved = build_state(_rpg)
_recs = recommend(_s, _f, played_games=_resolved, top_n=5)
show_recs(_recs)
print()
compliance_check(_recs, _f)


 1. 0.7414  The Witcher: Enhanced Edition  [?, no-cover]
 2. 0.7100  Red Dead Redemption 2  [2018, cover]
 3. 0.7035  Pathfinder: Wrath of the Righteous - Through the Ashes  [?, no-cover]
 4. 0.6958  Fallout 4  [2015, cover]
 5. 0.6740  Dark Souls II: Scholar of the First Sin  [?, no-cover]

Filter: {'year_min': 2010, 'platforms': ['PC']}
  The Witcher: Enhanced Edition: year=NaN(lenient); plat=['PC']
  Red Dead Redemption 2: year=2018; plat=['PC']
  Pathfinder: Wrath of the Righteous - Through the Ashes: year=NaN(lenient); plat=['PC']
  Fallout 4: year=2015; plat=['PC']
  Dark Souls II: Scholar of the First Sin: year=NaN(lenient); plat=['PC']


### Scenario 2 — same query, `profile_prefilter=False` (trust the policy / filter-only)



In [16]:
_recs = recommend(_s, _f, played_games=_resolved, top_n=5, profile_prefilter=False)
show_recs(_recs)
print()
compliance_check(_recs, _f)



 1. 0.7711  Half-Life 2  [?, no-cover]
 2. 0.7476  Battlefield Hardline: Getaway  [?, no-cover]
 3. 0.7434  Gothic II  [?, no-cover]
 4. 0.7433  Underworld  [?, no-cover]
 5. 0.7428  Cyberpunk 2077: Ultimate Edition  [?, no-cover]

Filter: {'year_min': 2010, 'platforms': ['PC']}
  Half-Life 2: year=NaN(lenient); plat=['PC']
  Battlefield Hardline: Getaway: year=NaN(lenient); plat=['PC']
  Gothic II: year=NaN(lenient); plat=['PC']
  Underworld: year=NaN(lenient); plat=['PC']
  Cyberpunk 2077: Ultimate Edition: year=NaN(lenient); plat=['PC']


### Scenario 3 — FPS fan · Nintendo Switch · 2019+ (does the year bind?)


In [17]:
_fps = ["DOOM Eternal", "Halo Infinite", "Titanfall 2"]
_f = {"year_min": 2019, "platforms": ["Nintendo Switch"]}
_s2, _r2 = build_state(_fps)
_recs = recommend(_s2, _f, played_games=_r2, top_n=5)
show_recs(_recs)
print()
compliance_check(_recs, _f)


 1. 0.8198  DOOM  [?, no-cover]
 2. 0.8009  Prince of Persia: The Lost Crown - Mask of Darkness  [?, no-cover]
 3. 0.7871  DOOM Eternal: The Ancient Gods Part Two  [?, no-cover]
 4. 0.7799  Dying Light: Platinum Edition  [?, no-cover]
 5. 0.7780  Quake Remastered  [?, no-cover]

Filter: {'year_min': 2019, 'platforms': ['Nintendo Switch']}
  DOOM: year=NaN(lenient); plat=['Nintendo Switch']
  Prince of Persia: The Lost Crown - Mask of Darkness: year=NaN(lenient); plat=['Nintendo Switch']
  DOOM Eternal: The Ancient Gods Part Two: year=NaN(lenient); plat=['Nintendo Switch']
  Dying Light: Platinum Edition: year=NaN(lenient); plat=['Nintendo Switch']
  Quake Remastered: year=NaN(lenient); plat=['Nintendo Switch']


### Scenario 4 — RPG fan · German-language filter (language leniency)


In [18]:
_f = {"language": "German"}
_recs = recommend(_s, _f, played_games=_resolved, top_n=5)
show_recs(_recs)
print()
compliance_check(_recs, _f)


 1. 0.7414  The Witcher: Enhanced Edition  [?, no-cover]
 2. 0.7100  Red Dead Redemption 2  [2018, cover]
 3. 0.6958  Fallout 4  [2015, cover]
 4. 0.6740  Dark Souls II: Scholar of the First Sin  [?, no-cover]
 5. 0.6650  Dark Souls II  [2014, cover]

Filter: {'language': 'German'}
  The Witcher: Enhanced Edition: no-language(lenient)
  Red Dead Redemption 2: no-language(lenient)
  Fallout 4: no-language(lenient)
  Dark Souls II: Scholar of the First Sin: no-language(lenient)
  Dark Souls II: no-language(lenient)


### Scenario 5 — cold start · PlayStation 5 · 2020+


In [19]:
_f = {"year_min": 2020, "platforms": ["PlayStation 5"]}
_recs = recommend(build_state([])[0], _f, played_games=[], top_n=5)
show_recs(_recs)
print()
compliance_check(_recs, _f)


 1. 0.8021  Tekken 2  [?, no-cover]
 2. 0.7990  Broken Sword II: The Smoking Mirror - Remastered  [?, no-cover]
 3. 0.7982  This War Of Mine: Complete Edition  [?, no-cover]
 4. 0.7929  Sega Ages: Phantasy Star Trilogy  [?, cover]
 5. 0.7839  Sonic Origins Plus  [?, no-cover]

Filter: {'year_min': 2020, 'platforms': ['PlayStation 5']}
  Tekken 2: year=NaN(lenient); plat=['PlayStation', 'PlayStation 5']
  Broken Sword II: The Smoking Mirror - Remastered: year=NaN(lenient); plat=['PlayStation', 'PlayStation 5']
  This War Of Mine: Complete Edition: year=NaN(lenient); plat=['PlayStation 5']
  Sega Ages: Phantasy Star Trilogy: year=NaN(lenient); no-platforms(lenient)
  Sonic Origins Plus: year=NaN(lenient); plat=['PlayStation 5']


### Why "Battlefield Hardline: Getaway" disappears under the default profile rerank

It's a **PC game with no release date**, so it *passes* the `2010+ / PC` filter
(NaN year → lenient pass; PC is in its platform list). It was **not** sacked by
the filter. What drops it is the candidate generator's **profile-cosine rerank +
`candidate_k` cap (default 30)**: it ranks **593rd** in raw-embedding similarity to the
RPG play history (it's a Battlefield/FPS title), far outside the kept candidates.

The reason it looks great in filter-only mode is that the two stages score by
*different* criteria: the **policy's predicted action** points near it (cosine
≈0.75), even though its raw similarity to the literal play history is only ≈0.51.
Play-history cosine (candidate gen) and learned policy action (reranker) disagree —
that's the whole point of the `profile_prefilter` knob.



In [20]:
_bf = "Battlefield Hardline: Getaway"
_row = name_to_row[_bf]
_f = {"year_min": 2010, "platforms": ["PC"]}
from app.candidate_generator import candidates as _cand
_filter_only = set(_cand(_f, played_games=None, k=None).tolist())
_profile_sorted = _cand(_f, played_games=_resolved, k=None).tolist()
print(f"{_bf}:")
print(f"  release_year = {index_df.iloc[_row]['release_year']}  (NaN -> lenient pass)")
print(f"  platforms    = {index_df.iloc[_row]['platforms']}")
print(f"  passes 2010+/PC filter?            {_row in _filter_only}")
print(f"  profile-rerank position (RPG)      {_profile_sorted.index(_row)} of {len(_profile_sorted)}  (default cap keeps 0..29)")
_profile = E[[name_to_row[g] for g in _resolved]].mean(0)
import torch as _t
with _t.no_grad():
    _act = policy(_t.from_numpy(_profile[None].astype(np.float32))).cpu().numpy().ravel()
_cos = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
print(f"  cos(game, RPG play-history profile) = {_cos(E[_row], _profile):.4f}")
print(f"  cos(game, policy predicted action)  = {_cos(E[_row], _act):.4f}")



Battlefield Hardline: Getaway:
  release_year = nan  (NaN -> lenient pass)
  platforms    = ['PlayStation 4', 'Xbox One', 'PC', 'Xbox 360', 'PlayStation 3']
  passes 2010+/PC filter?            True
  profile-rerank position (RPG)      593 of 13550  (default cap keeps 0..29)
  cos(game, RPG play-history profile) = 0.5102
  cos(game, policy predicted action)  = 0.7476


## Ad-hoc queries

Drop any game list here to probe the policy.


In [31]:
_games_ah = ["Marvel's Spider-Man", "Marvel's Spider-Man: Miles Morales", "Marvel's Spider-Man 2"]
_filter_ah = {"year_min": 2010, "platforms": ["Playstation 5"]}
_state_ah, _resolved_ah = build_state(_games_ah)
_recs_ah = recommend(_state_ah, _filter_ah, played_games=_resolved_ah, top_n=5, candidate_k=50)

show_recs(_recs_ah)
print()
compliance_check(_recs_ah, _filter_ah)

 1. 0.7814  Shadow Warrior 3: Definitive Edition  [?, no-cover]
 2. 0.7667  Skull and Bones  [?, no-cover]
 3. 0.7524  Horizon Zero Dawn Remastered  [?, no-cover]
 4. 0.7407  Final Fantasy VII Remake Intergrade  [?, no-cover]
 5. 0.7229  Metal Gear Solid Delta: Snake Eater  [2025, cover]

Filter: {'year_min': 2010, 'platforms': ['Playstation 5']}
  Shadow Warrior 3: Definitive Edition: year=NaN(lenient); plat=['PlayStation 5']
  Skull and Bones: year=NaN(lenient); plat=['PlayStation 5']
  Horizon Zero Dawn Remastered: year=NaN(lenient); plat=['PlayStation 5']
  Final Fantasy VII Remake Intergrade: year=NaN(lenient); plat=['PlayStation 5']
  Metal Gear Solid Delta: Snake Eater: year=2025; plat=['PlayStation 5']
